# Ch03 练习参考答案：Modified Policy Iteration

> 对比不同 k 的 modified policy iteration 在收敛速度和精度上的权衡。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import time
from utils import set_seed
from rlenvs import small_grid_5x5

set_seed(0)


def modified_policy_iteration(env, k=5, gamma=0.9, max_outer=100):
    nS, nA = env.nS, env.nA
    pi = np.full((nS, nA), 1.0 / nA)
    total_sweeps = 0
    for outer in range(max_outer):
        # k 次 sweep 的策略评估
        V = np.zeros(nS)
        for _ in range(k):
            Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
            new_V = (pi * Q).sum(axis=1)
            for s in range(nS):
                if env.is_terminal(s):
                    new_V[s] = 0.0
            V = new_V
            total_sweeps += 1
        # 贪心改进
        Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
        new_pi = np.zeros((nS, nA))
        new_pi[np.arange(nS), Q.argmax(axis=1)] = 1.0
        if np.array_equal(new_pi.argmax(axis=1), pi.argmax(axis=1)):
            return V, pi, outer + 1, total_sweeps
        pi = new_pi
    return V, pi, max_outer, total_sweeps


env = small_grid_5x5(seed=0)

# 真值（用纯策略迭代收敛）
def policy_iteration_exact(env, gamma=0.9, theta=1e-12):
    nS, nA = env.nS, env.nA
    pi = np.full((nS, nA), 1.0 / nA)
    while True:
        V = np.zeros(nS)
        while True:
            Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
            new_V = (pi * Q).sum(axis=1)
            for s in range(nS):
                if env.is_terminal(s):
                    new_V[s] = 0.0
            if np.abs(new_V - V).max() < theta:
                break
            V = new_V
        Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
        new_pi = np.zeros((nS, nA))
        new_pi[np.arange(nS), Q.argmax(axis=1)] = 1.0
        if np.array_equal(new_pi.argmax(axis=1), pi.argmax(axis=1)):
            return V, pi
        pi = new_pi

V_true, _ = policy_iteration_exact(env, gamma=0.9)

ks = [1, 2, 3, 5, 10, 20, 50, 100]
results = []
for k in ks:
    t0 = time.time()
    V, pi, n_outer, total_sweeps = modified_policy_iteration(env, k=k, gamma=0.9)
    t = time.time() - t0
    err = np.abs(V - V_true).max()
    results.append((k, n_outer, total_sweeps, err, t))
    print(f"k={k:<4}  outer iters={n_outer:<3}  total sweeps={total_sweeps:<4}  "
          f"V 误差={err:.2e}  时间={t*1000:.1f}ms")

print()
print("观察：")
print("- k=1（值迭代）通常 total sweeps 较大，但每次 sweep 便宜")
print("- k=5~10 通常 total sweeps 最少")
print("- k→∞ 接近纯策略迭代")